# Tarea 08: La cuenta en el aro, ahora con dinámica

**Semana 8** · Entrega: antes de la clase de la Semana 9

## Objetivos de la tarea

1. Plantear la dinámica de un sistema con `Particle` y energía potencial, y
   obtener su ecuación de movimiento con `LagrangesMethod`.
2. Obtener la misma ecuación con `KanesMethod` y comprobar que coinciden.
3. Encontrar las posiciones de equilibrio a partir de la ecuación de
   movimiento, e interpretar la velocidad de giro crítica.

In [11]:
import sympy as sp
import sympy.physics.mechanics as me
from IPython.display import Math

me.init_vprinting()

# Ya declarados: no los cambies, los tests los usan tal cual.
t = sp.Symbol("t")               # el tiempo de mechanics: sin suposiciones
N = me.ReferenceFrame("N")       # el marco del laboratorio
centro = me.Point("O")           # el centro del aro
centro.set_vel(N, 0)             # fijo en el laboratorio

## El sistema

Es el de la tarea 07: un aro de alambre de radio $R$, parado verticalmente,
**gira** alrededor de su diámetro vertical con velocidad angular constante
$\Omega$. Una cuenta de masa $m$ ensartada en el alambre se desliza por él
**sin fricción**; su posición la da el ángulo $\theta(t)$, medido desde el
punto más bajo. La gravedad $g$ apunta hacia $-\hat{n}_z$.

```
            n_z
             |
          .--+--.
        /    |    \
       |     O     |     ← el aro gira alrededor de n_z con Ω
        \    |  θ /
          '--+--●         ● la cuenta, de masa m
             |
             ↓ g
```

Los marcos son los mismos de la semana pasada:

- `N`: el laboratorio, con $\hat{n}_z$ vertical hacia arriba.
- `A`: el **aro**. Es $N$ girado un ángulo $\Omega t$ alrededor de `N.z`.
- `B`: la **cuenta**. Es $A$ girado un ángulo $\theta$ alrededor de `A.y`, el
  eje perpendicular al plano del aro.

La cuenta está en $-R\,\hat{b}_z$.

La semana pasada era cinemática: dado $\theta(t)$, dónde está la cuenta. Esta
semana la pregunta es la contraria: **qué $\theta(t)$ sigue la cuenta** por
sí sola. Todo lo que necesitas está en los dos notebooks de la semana 8.

## Ejercicio 1: el sistema

Completa la celda con estos nombres exactos, que son los que usa
`tests/test_tarea.py`:

- `radio`, `omega`, `masa` y `gravedad`: símbolos **positivos**, con nombres
  de símbolo `"R"`, `"Omega"`, `"m"` y `"g"`.
- `theta`: el ángulo de la cuenta, como `dynamicsymbol` de nombre `"theta"`.
- `A` y `B`: los marcos del aro y de la cuenta, orientados con `orient_axis`.
- `cuenta`: un `me.Point` ubicado respecto a `centro` en $-R\,\hat{b}_z$.
- `particula`: una `me.Particle` de masa `masa` en `cuenta`, con su
  `potential_energy` gravitacional. Ojo: aquí la vertical es $\hat{n}_z$, no
  $\hat{n}_y$ como en clase.

Antes de seguir, muestra la energía potencial con `sp.Eq`: debe salir
$-mgR\cos\theta$. ¿Por qué es mínima en $\theta = 0$?

In [12]:
# TODO: completa el ejercicio 1
radio = sp.Symbol("R", positive=True)
omega = sp.Symbol("Omega", positive=True)
masa = sp.Symbol("m", positive=True)
gravedad = sp.Symbol("g", positive=True)
theta = me.dynamicsymbols("theta")

A = me.ReferenceFrame("A")
A.orient_axis(N, N.z, omega * t)

B = me.ReferenceFrame("B")
B.orient_axis(A, A.y, theta)

cuenta = me.Point("cuenta")
cuenta.set_pos(centro, -radio * B.z)

particula = me.Particle("particula", cuenta, masa)
# La vertical es N.z, por lo que la altura es el producto punto con N.z
particula.potential_energy = masa * gravedad * me.dot(cuenta.pos_from(centro), N.z)

# Mostrar la energía potencial
display(sp.Eq(sp.Symbol("V"), particula.potential_energy))

## Ejercicio 2: Lagrange

- `lagrangiano`: el Lagrangiano de la partícula, con `me.Lagrangian`.
  Muéstralo: su energía cinética es la rapidez al cuadrado del ejercicio 4 de
  la tarea 07, por $m/2$.
- `metodo_lagrange`: un `LagrangesMethod` con la coordenada `theta`, con sus
  ecuaciones ya generadas.
- `ecuacion_lagrange`: la **única** ecuación que genera (el elemento `[0]` de
  la matriz), mostrada con `sp.Eq(..., 0)`.

In [13]:
# TODO: completa el ejercicio 2

# Es necesario definir primero la velocidad de la cuenta en el marco N
# (tomando en cuenta que 'centro' ya estaba fijo en N en las celdas previas)
cuenta.set_vel(N, cuenta.pos_from(centro).dt(N))

# 1. Calcular el Lagrangiano
lagrangiano = sp.simplify(me.Lagrangian(N, particula))

# 2. Plantear el método de Lagrange
metodo_lagrange = me.LagrangesMethod(lagrangiano, [theta])
metodo_lagrange.form_lagranges_equations() # Genera las ecuaciones internamente

# 3. Extraer la ecuación de movimiento (único elemento de la matriz columna)
ecuacion_lagrange = sp.simplify(metodo_lagrange.eom[0])

# Mostrar el Lagrangiano y la ecuación de movimiento igualada a cero
display(Math(r"\mathcal{L} = " + sp.latex(lagrangiano)))
display(sp.Eq(ecuacion_lagrange, 0))

<IPython.core.display.Math object>

## Ejercicio 3: Kane

La misma ecuación, por la otra ruta. Hazlo **después** del ejercicio 2: aquí
le cambias a `B` la forma de escribir su velocidad angular, y eso cambia las
velocidades que calcula `mechanics` de ahí en adelante.

- `u`: la velocidad generalizada, como `dynamicsymbol` de nombre `"u"`.
- Fija la velocidad angular de `B` **respecto a `A`** en términos de `u`, con
  `set_ang_vel`, y recalcula la velocidad de `cuenta` en `N` con
  `v2pt_theory` (`centro` y `cuenta` están fijos en `B`).
- `metodo_kane`: un `KanesMethod` con `theta`, `u` y su ecuación cinemática.
  La única carga es el peso de la cuenta.
- `ecuacion_kane`: la única ecuación, el elemento `[0]` de $F_r + F_r^*$.

Compruébalo tú: el segundo renglón de `metodo_kane.rhs()` y el de
`metodo_lagrange.rhs()`, después de sustituir $\dot\theta \to u$, deben
diferir en cero.

In [14]:
# TODO: completa el ejercicio 3
u = me.dynamicsymbols("u")

# 1. Fijar la velocidad angular de B respecto a A explícitamente usando u
B.set_ang_vel(A, u * A.y)

# 2. Recalcular la velocidad de la cuenta en N usando el teorema de 2 puntos
# (centro y cuenta se mantienen en posiciones fijas vistas desde el marco B)
cuenta.v2pt_theory(centro, N, B)

# 3. Plantear el método de Kane
ecuacion_cinematica = [u - theta.diff(t)]
metodo_kane = me.KanesMethod(N, q_ind=[theta], u_ind=[u], kd_eqs=ecuacion_cinematica)

# La única carga externa es el peso (apuntando en -N.z)
cargas = [(cuenta, -masa * gravedad * N.z)]
cuerpos = [particula]

fr, frstar = metodo_kane.kanes_equations(cuerpos, cargas)

# 4. Extraer la ecuación de Kane (elemento 0 del vector Fr + Fr*)
ecuacion_kane = sp.simplify((fr + frstar)[0])

# Mostrar la ecuación
display(Math(r"\text{Ecuación de Kane: }"))
display(sp.Eq(ecuacion_kane, 0))

# 5. Comprobación cruzada con Lagrange
rhs_lagrange = metodo_lagrange.rhs()
rhs_kane = metodo_kane.rhs()

# Comparamos las aceleraciones (el segundo elemento [1] de los RHS)
diferencia = sp.simplify(rhs_kane[1] - rhs_lagrange[1].subs({theta.diff(t): u}))
print("¿Las ecuaciones de Kane y Lagrange coinciden?:", diferencia == 0)

<IPython.core.display.Math object>

¿Las ecuaciones de Kane y Lagrange coinciden?: True


## Ejercicio 4: equilibrios

- `aceleracion_theta`: $\ddot\theta$ despejada, es decir, el segundo renglón
  de `metodo_lagrange.rhs()`, simplificado. Debe depender solo de $\theta$.
- `equilibrio_lateral`: una posición de equilibrio es un ángulo donde la
  cuenta, puesta en reposo, se queda en reposo: $\ddot\theta = 0$. Hay dos
  obvias, $\theta = 0$ y $\theta = \pi$, y un par más que no lo es. Para
  encontrarlas, cambia $\theta(t)$ por un símbolo común,
  `theta_0 = sp.Symbol("theta_0")`, y usa `sp.solve`. Guarda en
  `equilibrio_lateral` la solución que queda entre $0$ y $\pi/2$.
- `omega_critica`: `equilibrio_lateral` solo existe si el argumento de su
  arcocoseno es menor que 1. Encuentra la $\Omega$ en la que vale
  exactamente 1, resolviendo con `sp.solve`.

Todo con cantidades exactas: nada de decimales.

In [15]:
# TODO: completa el ejercicio 4
# 1. Extraer la aceleración (el segundo elemento del vector de estado rhs)
aceleracion_theta = sp.simplify(metodo_lagrange.rhs()[1])

# 2. Encontrar los puntos de equilibrio
theta_0 = sp.Symbol("theta_0")
aceleracion_estatica = aceleracion_theta.subs(theta, theta_0)
soluciones_equilibrio = sp.solve(aceleracion_estatica, theta_0)

# Filtrar la solución que contiene el arcocoseno (la posición lateral entre 0 y pi/2)
equilibrio_lateral = [sol for sol in soluciones_equilibrio if sol.has(sp.acos)][0]

# 3. Encontrar la omega crítica
# En lugar de navegar por el árbol de la expresión, usamos trigonometría:
# La solución existe cuando su coseno alcanza el límite de 1.
coseno_equilibrio = sp.simplify(sp.cos(equilibrio_lateral))

# Resolvemos cos(theta_eq) - 1 = 0 para Omega
omega_critica = sp.solve(coseno_equilibrio - 1, omega)[0]

# Mostrar los resultados
display(Math(r"\ddot{\theta} = " + sp.latex(aceleracion_theta)))
display(Math(r"\theta_{eq} = " + sp.latex(equilibrio_lateral)))
display(Math(r"\Omega_{crit} = " + sp.latex(omega_critica)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Discusión

_Contesta aquí, en uno o dos párrafos. Esta parte no la califica el
autograding: la revisa el asistente._

1. En el ejercicio 1 escribiste la energía potencial, pero en el 3 no la
   usaste. ¿Cómo entró la gravedad en el método de Kane?
2. Si el aro gira más lento que `omega_critica`, el único equilibrio abajo es
   $\theta = 0$. Si gira más rápido, aparece `equilibrio_lateral`. Explica con
   la "fuerza centrífuga" de la tarea 07 por qué girar rápido saca a la cuenta
   del fondo.
3. $\theta = \pi$ (la cuenta hasta arriba) también cumple $\ddot\theta = 0$.
   ¿Es un equilibrio en el que esperarías encontrar a la cuenta? ¿Qué pasa si
   la empujas un poquito?

_(Escribe tu respuesta aquí.)_

1. A diferencia de Lagrange, el método de Kane no utiliza funciones escalares de energía (como el potencial $V$). En Kane, la gravedad entró directamente como una **fuerza activa vectorial**. En el código, esto ocurrió cuando definimos explícitamente el peso de la cuenta como el vector `-masa * gravedad * N.z` y lo incluimos en la lista de `cargas` aplicadas sobre la partícula para calcular las fuerzas generalizadas $F_r$.

2. En el fondo del aro ($\theta = 0$), la cuenta se encuentra exactamente sobre el eje de giro, por lo que la fuerza centrífuga es cero. Sin embargo, al perturbarla ligeramente, aparecen dos efectos opuestos a lo largo del alambre: la gravedad que intenta regresarla al fondo, y la "fuerza centrífuga" que la empuja hacia afuera. Si el aro gira más rápido que $\Omega_{crit}$, la componente tangencial de la fuerza centrífuga (proporcional a $\Omega^2$) supera a la gravedad cerca del origen. Esto vuelve inestable el fondo y empuja a la cuenta hacia arriba hasta alcanzar el `equilibrio_lateral`, el punto exacto donde ambas fuerzas tangenciales se contrarrestan.

3. El punto $\theta = \pi$ (hasta arriba del aro) es un equilibrio **inestable**, por lo que físicamente no esperaríamos encontrar a la cuenta reposando ahí. Si la perturbamos o empujamos un poquito, sale del eje de rotación: a partir de ahí, tanto la gravedad (que la jala hacia abajo) como la fuerza centrífuga (que la empuja hacia afuera del eje) actuarán a favor de alejarla de la cima, haciéndola caer aceleradamente hacia los equilibrios estables en la parte inferior.

## Resumen

Entrega esta tarea por **Pull Request dentro de tu fork**, como las anteriores.
El paso a paso está en [`docs/git-guia.md`](../../docs/git-guia.md), y el
checklist completo en [`README.md`](README.md).

Antes de entregar, corre el notebook desde un kernel limpio
(`Kernel → Restart & Run All`) y confirma que llega hasta el final sin errores.
Al hacer push, GitHub Actions corre los tests y deja un check ✅ o ❌ en tu
Pull Request.